In [1]:
import pandas as pd
import re
from dateutil import parser
from sqlalchemy import create_engine

year = 2025
df = pd.read_csv('../amazon_india_2025.csv', low_memory=False)

def clean_price(val):
    val = str(val).replace('₹', '').replace(',', '').strip()
    try: return float(val)
    except: return None

def clean_rating(val):
    if pd.isna(val): return None
    val = str(val).lower().strip()
    if 'star' in val: return float(re.search(r'\d+\.?\d*', val).group())
    if '/' in val: a, b = val.split('/'); return round(float(a)/float(b)*5, 2)
    try: return float(val)
    except: return None

def clean_date(val):
    try: return parser.parse(val, dayfirst=True).strftime('%Y-%m-%d')
    except: return None

def clean_delivery(val):
    val = str(val).strip().lower()
    if val == 'same day': return 0
    if val == 'express': return 1
    if '-' in val:
        try:
            a, b = val.split('-')
            return round((float(a) + float(b.split()[0])) / 2)
        except: return None
    try:
        val = int(float(val))
        if val < 0: return None
        if val > 30: return None
        return val
    except: return None


true_values = ['True', 'TRUE', '1', 'Yes']
manual_mapping = {'Bombay': 'Mumbai', 'Madras': 'Chennai', 'Bengaluru': 'Bangalore', 'Calcutta': 'Kolkata'}
category_mapping = {'ELECTRONICS': 'Electronics', 'Electronics & Accessories': 'Electronics', 'Electronic': 'Electronics', 'Electronicss': 'Electronics'}
payment_mapping = {
    'Cash on Delivery': 'COD', 'C.O.D': 'COD',
    'CREDIT CARD': 'Credit Card', 'CC': 'Credit Card',
    'DEBIT CARD': 'Debit Card',
    'NET BANKING': 'Net Banking',
    'UPI': 'UPI', 'PhonePe': 'UPI', 'GooglePay': 'UPI', 'GPay': 'UPI'
}

df['original_price_inr'] = df['original_price_inr'].apply(clean_price)
df['product_rating'] = df['product_rating'].apply(clean_rating)
df['product_rating'] = df.groupby('category')['product_rating'].transform(lambda x: x.fillna(x.median()))
df['product_rating'] = df['product_rating'].fillna(df['product_rating'].median())
df['customer_rating'] = df['customer_rating'].apply(clean_rating)
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())
df['order_date'] = df['order_date'].apply(clean_date)
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_days'] = df['delivery_days'].apply(clean_delivery)
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['customer_age_group'] = df['customer_age_group'].fillna(df['customer_age_group'].mode()[0])
df['delivery_charges'] = df['delivery_charges'].fillna(0)
df['festival_name'] = df['festival_name'].fillna('No Festival')
df['is_prime_member'] = False
df['is_prime_eligible'] = df['is_prime_eligible'].isin(true_values)
df['is_festival_sale'] = df['is_festival_sale'].isin(true_values)
df['customer_city'] = df['customer_city'].str.strip().str.title().replace(manual_mapping)
df['category'] = df['category'].replace(category_mapping)
df['payment_method'] = df['payment_method'].replace(payment_mapping)

df['original_price_inr'] = df.apply(
    lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100)
    if pd.isna(row['original_price_inr']) or row['original_price_inr'] > row['discounted_price_inr'] * 10
    else row['original_price_inr'], axis=1
)
df.loc[df['original_price_inr'] < df['discounted_price_inr'], 'original_price_inr'] = df.apply(
    lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100)
    if row['original_price_inr'] < row['discounted_price_inr']
    else row['original_price_inr'], axis=1
)

df = df.drop_duplicates(subset=['customer_id', 'product_id', 'order_date', 'final_amount_inr'], keep='first')

print(df.isnull().sum())

df.to_csv(f'2025_amazon.csv', index=False)
engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/amazon_db?unix_socket=/Applications/XAMPP/xamppfiles/var/mysql/mysql.sock')
df.to_sql(f'transactions_{year}', engine, if_exists='replace', index=False)


transaction_id            0
order_date                0
customer_id               0
product_id                0
product_name              0
category                  0
subcategory               0
brand                     0
original_price_inr        0
discount_percent          0
discounted_price_inr      0
quantity                  0
subtotal_inr              0
delivery_charges          0
final_amount_inr          0
customer_city             0
customer_state            0
customer_tier             0
customer_spending_tier    0
customer_age_group        0
payment_method            0
delivery_days             0
delivery_type             0
is_prime_member           0
is_festival_sale          0
festival_name             0
customer_rating           0
return_status             0
order_month               0
order_year                0
order_quarter             0
product_weight_kg         0
is_prime_eligible         0
product_rating            0
dtype: int64


77000

In [2]:
import glob
files = glob.glob('amazon_india_*_cleaned.csv')
print(files)

[]


In [3]:
import os
print(os.getcwd())  
print(os.listdir()) 

/Users/Sanjay/Desktop/GUVI/Project 2/CODE
['2023_amazon.csv', 'cleaning_2017.ipynb', '2019_amazon.csv', '2024_amazon.csv', 'cleaning_2016.ipynb', '2021_amazon.csv', '2016_amazon.csv', 'Untitled-2.ipynb', 'cleaning_2018.ipynb', 'cleaning_2025.ipynb', '2015_amazon.csv', '2022_amazon.csv', '2018_amazon.csv', 'cleaning_2023.ipynb', 'cleaning_2021.ipynb', '2020_amazon.csv', '2017_amazon.csv', 'cleaning_2024.ipynb', 'cleaning_2019.ipynb', 'cleaning_2020.ipynb', '2025_amazon.csv', 'cleaning_2022.ipynb']


In [4]:
import glob
all_cities = set()
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file)
    cities = df['customer_city'].unique()
    all_cities.update(cities)

print(sorted(all_cities))
print(f"Total unique cities: {len(all_cities)}")


/var/folders/8f/6_l0xx0x2s7gs572hcv0rpsh0000gp/T/ipykernel_7453/4292356286.py:4: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


['Ahmedabad', 'Aligarh', 'Allahabad', 'Bangalore', 'Banglore', 'Bareilly', 'Bengalore', 'Bhubaneswar', 'Chandigarh', 'Chenai', 'Chennai', 'Coimbatore', 'Delhi', 'Delhi Ncr', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumba', 'Mumbai', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']
Total unique cities: 36


In [5]:
fix_mapping = {
    'Banglore': 'Bangalore',
    'Bengalore': 'Bangalore',
    'Chenai': 'Chennai',
    'Delhi Ncr': 'Delhi',
    'Mumba': 'Mumbai',
    'New Delhi': 'Delhi',
}

for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    df['customer_city'] = df['customer_city'].replace(fix_mapping)
    df.to_csv(file, index=False)
    print(f"✅ Fixed {file}")

✅ Fixed 2023_amazon.csv
✅ Fixed 2019_amazon.csv
✅ Fixed 2024_amazon.csv
✅ Fixed 2021_amazon.csv
✅ Fixed 2016_amazon.csv
✅ Fixed 2015_amazon.csv
✅ Fixed 2022_amazon.csv
✅ Fixed 2018_amazon.csv
✅ Fixed 2020_amazon.csv
✅ Fixed 2017_amazon.csv
✅ Fixed 2025_amazon.csv


In [6]:
import glob
import pandas as pd

all_data = []
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    all_data.append(df)

# combine all years
combined = pd.concat(all_data, ignore_index=True)

# check all columns
print("=== SHAPE ===")
print(combined.shape)

print("\n=== MISSING VALUES ===")
print(combined.isnull().sum())

print("\n=== CATEGORY ===")
print(combined['category'].unique())

print("\n=== PAYMENT METHOD ===")
print(combined['payment_method'].unique())

print("\n=== IS_FESTIVAL_SALE ===")
print(combined['is_festival_sale'].value_counts())

print("\n=== IS_PRIME_MEMBER ===")
print(combined['is_prime_member'].value_counts())

print("\n=== IS_PRIME_ELIGIBLE ===")
print(combined['is_prime_eligible'].value_counts())

print("\n=== DELIVERY_TYPE ===")
print(combined['delivery_type'].unique())

print("\n=== CUSTOMER_TIER ===")
print(combined['customer_tier'].unique())

print("\n=== CUSTOMER_AGE_GROUP ===")
print(combined['customer_age_group'].unique())

=== SHAPE ===
(1121999, 34)

=== MISSING VALUES ===
transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr            0
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name             22459
customer_rating            9926
return_status                 0
order_month                   0
order_year          

In [7]:
print("=== MISSING VALUES ===")
print(combined.isnull().sum())

print("\n=== CATEGORY ===")
print(combined['category'].unique())

print("\n=== PAYMENT METHOD ===")
print(combined['payment_method'].unique())

print("\n=== IS_FESTIVAL_SALE ===")
print(combined['is_festival_sale'].value_counts())

print("\n=== IS_PRIME_MEMBER ===")
print(combined['is_prime_member'].value_counts())

print("\n=== IS_PRIME_ELIGIBLE ===")
print(combined['is_prime_eligible'].value_counts())

print("\n=== DELIVERY_TYPE ===")
print(combined['delivery_type'].unique())


=== MISSING VALUES ===
transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr            0
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name             22459
customer_rating            9926
return_status                 0
order_month                   0
order_year                    0
order_quarter    

In [8]:
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())
    df.to_csv(file, index=False)
    print(f"✅ Fixed {file}")
    

✅ Fixed 2023_amazon.csv
✅ Fixed 2019_amazon.csv
✅ Fixed 2024_amazon.csv
✅ Fixed 2021_amazon.csv
✅ Fixed 2016_amazon.csv


TypeError: Cannot convert ['5.0' '4.5' nan ... '5.0' '5.0' nan] to numeric

In [9]:
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    df['customer_rating'] = pd.to_numeric(df['customer_rating'], errors='coerce')
    df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())
    df.to_csv(file, index=False)
    print(f"✅ Fixed {file}")
    

✅ Fixed 2023_amazon.csv
✅ Fixed 2019_amazon.csv
✅ Fixed 2024_amazon.csv
✅ Fixed 2021_amazon.csv
✅ Fixed 2016_amazon.csv
✅ Fixed 2015_amazon.csv
✅ Fixed 2022_amazon.csv
✅ Fixed 2018_amazon.csv
✅ Fixed 2020_amazon.csv
✅ Fixed 2017_amazon.csv
✅ Fixed 2025_amazon.csv


In [10]:
fix_mapping = {
    'Banglore': 'Bangalore',
    'Bengalore': 'Bangalore',
    'Chenai': 'Chennai',
    'Delhi Ncr': 'Delhi',
    'Mumba': 'Mumbai',
    'New Delhi': 'Delhi',
}

for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    df['customer_city'] = df['customer_city'].replace(fix_mapping)
    df.to_csv(file, index=False)
    print(f"✅ Fixed {file}")

✅ Fixed 2023_amazon.csv
✅ Fixed 2019_amazon.csv
✅ Fixed 2024_amazon.csv
✅ Fixed 2021_amazon.csv
✅ Fixed 2016_amazon.csv
✅ Fixed 2015_amazon.csv
✅ Fixed 2022_amazon.csv
✅ Fixed 2018_amazon.csv
✅ Fixed 2020_amazon.csv
✅ Fixed 2017_amazon.csv
✅ Fixed 2025_amazon.csv


In [11]:
all_cities = set()
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    all_cities.update(df['customer_city'].unique())

print(sorted(all_cities))
print(f"Total unique cities: {len(all_cities)}")

['Ahmedabad', 'Aligarh', 'Allahabad', 'Bangalore', 'Bareilly', 'Bhubaneswar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumbai', 'Nagpur', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']
Total unique cities: 30


In [12]:
combined = pd.concat([pd.read_csv(f, low_memory=False) for f in glob.glob('*_amazon.csv')], ignore_index=True)
print(f"Total rows: {len(combined)}")
print(f"Missing values:\n{combined.isnull().sum()}")




Total rows: 1121999
Missing values:
transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr            0
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name             22459
customer_rating               0
return_status                 0
order_month                   0
order_year                    0
orde

In [13]:
for file in glob.glob('*_amazon.csv'):
    df = pd.read_csv(file, low_memory=False)
    null_count = df['festival_name'].isna().sum()
    if null_count > 0:
        print(f"{file}: {null_count} missing festival names")

2015_amazon.csv: 22459 missing festival names


In [14]:
from sqlalchemy import create_engine

engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/amazon_db?unix_socket=/Applications/XAMPP/xamppfiles/var/mysql/mysql.sock')

for file in glob.glob('*_amazon.csv'):
    year = file.split('_')[0]  # extracts year from filename
    df = pd.read_csv(file, low_memory=False)
    df.to_sql(f'transactions_{year}', engine, if_exists='replace', index=False)
    print(f"✅ Saved transactions_{year} to MySQL!")

print("\n🎉 All 10 years saved to MySQL!")

✅ Saved transactions_2023 to MySQL!
✅ Saved transactions_2019 to MySQL!
✅ Saved transactions_2024 to MySQL!
✅ Saved transactions_2021 to MySQL!
✅ Saved transactions_2016 to MySQL!
✅ Saved transactions_2015 to MySQL!
✅ Saved transactions_2022 to MySQL!
✅ Saved transactions_2018 to MySQL!
✅ Saved transactions_2020 to MySQL!
✅ Saved transactions_2017 to MySQL!
✅ Saved transactions_2025 to MySQL!

🎉 All 10 years saved to MySQL!


In [1]:
import pandas as pd
import os

# Folder where your yearly CSVs are stored
folder_path = "/Users/Sanjay/Desktop/GUVI/Project 2/CODE/"

# Get all CSV files
files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

# Empty list to store dataframes
df_list = []

for file in files:
    file_path = os.path.join(folder_path, file)
    
    df = pd.read_csv(file_path)
    
    # Optional: extract year from filename
    year = file.split("_")[0]
    df['year'] = int(year)
    
    df_list.append(df)

# Combine all
combined = pd.concat(df_list, ignore_index=True)

# Save final dataset
combined.to_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/combined.csv", index=False)

print("✅ Combined dataset created successfully!")
print(combined.shape)

✅ Combined dataset created successfully!
(1121999, 35)


In [2]:
import pandas as pd

# Load your combined dataset
combined = pd.read_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/combined.csv")

# Select product-related columns
products = combined[[
    'product_id',
    'product_name',
    'category',
    'subcategory',
    'brand',
    'discounted_price_inr',
    'product_rating',
    'product_weight_kg',
    'is_prime_eligible'
]].drop_duplicates()

# Save as products.csv
products.to_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/products.csv", index=False)

print("✅ products.csv created")

✅ products.csv created


In [3]:
import pandas as pd

combined = pd.read_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/combined.csv")

products = combined[[
    'product_id',
    'product_name',
    'category',
    'subcategory',
    'brand',
    'discounted_price_inr',
    'product_rating',
    'product_weight_kg',
    'is_prime_eligible'
]].drop_duplicates()

products.to_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/products.csv", index=False)

In [2]:
import pandas as pd

In [5]:
df = pd.read_csv("/Users/Sanjay/Desktop/GUVI/amazon_dashboard/products.csv")


In [6]:
df.columns

Index(['product_id', 'product_name', 'category', 'subcategory', 'brand',
       'discounted_price_inr', 'product_rating', 'product_weight_kg',
       'is_prime_eligible'],
      dtype='object')

In [7]:
df = pd.read_csv("/Users/Sanjay/Desktop/GUVI/Project 2/amazon_india_products_catalog.csv")


In [8]:
df.columns

Index(['product_id', 'product_name', 'category', 'subcategory', 'brand',
       'base_price_2015', 'weight_kg', 'rating', 'is_prime_eligible',
       'launch_year', 'model'],
      dtype='object')